# NFI / NIR Metrics and fsQCA Calibration

Computes the **Net Flexibility Incentive (NFI)** and **Net Incentive Ratio (NIR)** from the
solver output CSVs, and exports the **fsQCA-ready condition/outcome table** for the Germany
case set (Archetypes 3–8 × 7 DSOs = 42 cases).

## Metric definitions (Chapter 4, Section 4.3)

| Metric | Formula | Interpretation |
|---|---|---|
| **NFI_eur** | `TCoE_noflex − TCoE_tcoeflex` | Absolute annual bill saving from full-stack opt (€/yr); always defined |
| **NIR** | `(TCoE_noflex − TCoE_dtflex) / (TCoE_noflex − TCoE_tcoeflex)` | Fraction of full-stack saving captured by spot-only opt |

NIR = 1 → perfect alignment; NIR < 1 → dilution; NIR > 1 → amplification. Undefined for A1/A2 (no flex DOF).

## fsQCA design (Chapter 4, Section 4.8)

| Condition | Source | Type |
|---|---|---|
| `BSS` | archetype ∈ {3,8} | Binary |
| `HP` | archetype ∈ {4,6,8} | Binary |
| `EV` | archetype ∈ {5,7,8} | Binary |
| `MOD3_SIGNAL` | HT − NT spread (ct/kWh) from DSO tariff | Fuzzy, calibrated from data distribution |
| `DSO_VOL_LEVEL` | Arbeitspreis (ct/kWh) from DSO tariff | Fuzzy, calibrated from data distribution |
| **LOW_NIR** (outcome) | NIR with inverted direction | Fuzzy, anchored at 25th/50th/75th pctile of NIR |

## Inputs: `outputs/results_*.csv`
## Outputs: `outputs/nfi_nir_germany_2026.csv`, `outputs/fsqca_conditions_germany_2026.csv`
## Thesis reference: Chapter 5, Sections 5.3 and 5.6

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

def find_repo_root(marker='README.md'):
    p = Path('__file__').resolve().parent
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError('Repo root not found')

REPO_ROOT  = find_repo_root()
INPUTS     = REPO_ROOT / 'inputs'
ANALYSIS   = Path('__file__').resolve().parent  # save outputs next to this notebook
print(f'Repo root: {REPO_ROOT}')


Repo root: /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub


## Step 1 — Load and consolidate all solver outputs

In [2]:
# All eight archetype result files; each contains rows for 3 strategies × 7 DSOs
csv_files = sorted((REPO_ROOT / 'outputs').glob('results_*.csv'))
print('CSV files found:', [f.name for f in csv_files])

raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f'Total rows loaded: {len(raw)}')
print('Archetypes:', sorted(raw['household_archetype'].unique()))
print('DSOs:      ', sorted(raw['dso_id'].unique()))
print('Strategies:', sorted(raw['strategy'].unique()))


CSV files found: ['results_base_2026.csv', 'results_base_ev_2026.csv', 'results_base_hp_2026.csv', 'results_base_pv_2026.csv', 'results_base_pv_bss_2026.csv', 'results_base_pv_bss_hp_ev_2026.csv', 'results_base_pv_ev_2026.csv', 'results_base_pv_hp_2026.csv']
Total rows loaded: 168
Archetypes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
DSOs:       ['Bayernwerk', 'E.DIS', 'MITNETZ STROM', 'Netze BW', 'SH Netz', 'Stromnetz Berlin', 'Westnetz']
Strategies: ['dt_flex', 'no_flex', 'tcoe_flex']


## Step 2 — Pivot to wide format and compute NFI / NIR

Each (archetype, DSO) pair needs TCoE for all three strategies on one row before we can compute
the ratios. We pivot on strategy, then apply the formulas.

In [3]:
# Pivot: one row per (archetype, DSO), three TCoE columns
wide = raw.pivot_table(
    index=['household_archetype', 'dso_id'],
    columns='strategy',
    values='total_tcoe_eur',
    aggfunc='first'
).reset_index()
wide.columns.name = None

# Rename for clarity
wide = wide.rename(columns={
    'no_flex'  : 'TCoE_noflex',
    'dt_flex'  : 'TCoE_dtflex',
    'tcoe_flex': 'TCoE_tcoeflex',
})

# NFI_eur: absolute annual saving from full-stack optimisation vs. no-flex [EUR/yr].
# Defined for all archetypes including those with negative TCoE (PV export earners).
# A positive value always means tcoe_flex reduces the bill (or increases net income).
wide['NFI_eur'] = (wide['TCoE_noflex'] - wide['TCoE_tcoeflex']).round(2)

# Absolute dt_flex saving (EUR/yr)
wide['saving_dtflex_eur'] = (wide['TCoE_noflex'] - wide['TCoE_dtflex']).round(2)

# NIR: fraction of full-stack saving captured by spot-only DT-flex
# Denominator is zero for archetypes without any flexibility DOF (A1, A2)
denom = wide['TCoE_noflex'] - wide['TCoE_tcoeflex']
no_flex_archetypes = [1, 2]   # A1 (Base) and A2 (Base+PV) have no dispatch DOF
wide['NIR'] = np.where(
    wide['household_archetype'].isin(no_flex_archetypes) | (denom.abs() < 1e-6),
    np.nan,
    ((wide['TCoE_noflex'] - wide['TCoE_dtflex']) / denom).round(4)
)

print(wide[['household_archetype','dso_id','TCoE_noflex','TCoE_dtflex','TCoE_tcoeflex',
            'NFI_eur','NIR']].to_string(index=False))


 household_archetype           dso_id  TCoE_noflex  TCoE_dtflex  TCoE_tcoeflex  NFI_eur    NIR
                   1       Bayernwerk      1224.96      1224.96        1224.96     0.00    NaN
                   1            E.DIS      1239.06      1239.06        1239.06     0.00    NaN
                   1    MITNETZ STROM      1279.70      1279.70        1279.70     0.00    NaN
                   1         Netze BW      1360.26      1360.26        1360.26     0.00    NaN
                   1          SH Netz      1310.58      1310.58        1310.58     0.00    NaN
                   1 Stromnetz Berlin      1294.11      1294.11        1294.11     0.00    NaN
                   1         Westnetz      1460.82      1460.82        1460.82     0.00    NaN
                   2       Bayernwerk       267.22       267.22         267.22     0.00    NaN
                   2            E.DIS       264.94       264.94         264.94     0.00    NaN
                   2    MITNETZ STROM       287.23

## Step 3 — Summary statistics

Quick sanity check: NFI/NIR distribution by archetype (averaged across DSOs).

In [4]:
summary = (
    wide.groupby('household_archetype')
    [['NFI_eur','NIR']]
    .agg(['mean','min','max'])
    .round(3)
)
print('NFI / NIR summary by archetype (mean | min | max across 7 DSOs):')
display(summary)

nir_flex = wide[wide['household_archetype'].isin(range(3,9))]['NIR'].dropna()
q25, q50, q75 = nir_flex.quantile([0.25, 0.50, 0.75]).values
print(f'\nNIR distribution (N=42 flexible cases): '
      f'p25={q25:.3f} | median={q50:.3f} | p75={q75:.3f} | '
      f'min={nir_flex.min():.3f} | max={nir_flex.max():.3f}')


NFI / NIR summary by archetype (mean | min | max across 7 DSOs):


NFI_eur                      NIR              
                         mean      min      max   mean    min    max
household_archetype                                                 
1                       0.000     0.00     0.00    NaN    NaN    NaN
2                       0.000     0.00     0.00    NaN    NaN    NaN
3                     176.853   176.18   177.44  0.468  0.335  0.566
4                     340.639   193.10   517.51  0.465  0.276  0.767
5                     553.689   493.22   666.85  0.793  0.652  0.884
6                     568.003   402.59   755.61  0.294  0.214  0.402
7                     763.610   663.64   885.45  0.578  0.495  0.661
8                    1322.469  1059.15  1619.25  0.491  0.377  0.623


NIR distribution (N=42 flexible cases): p25=0.405 | median=0.493 | p75=0.595 | min=0.214 | max=0.884


## Step 4 — fsQCA condition construction and calibration

**Direct logistic calibration** (Ragin 2008) maps each raw score to a [0,1] fuzzy membership
using three empirical anchors: full non-membership threshold, crossover (0.5), and full
membership threshold.

```
fuzzy = 1 / (1 + exp(b × (x − crossover)))
```

where `b` is derived so that the full-membership anchor maps to 0.95 and the full
non-membership anchor maps to 0.05:

```
b = log(19) / (crossover − full_membership_anchor)   [for 'high is good' direction]
```

For **LOW_NIR** the direction is inverted: lower NIR → higher membership.

In [5]:
def calibrate_fuzzy(x, full_in, crossover, full_out):
    """Direct logistic calibration (Ragin 2008) to fuzzy membership in [0,1].

    full_in  : raw score that should map to ~0.95 (full set membership)
    crossover: raw score that maps to exactly 0.5
    full_out : raw score that should map to ~0.05 (full set non-membership)
    Direction: higher x → higher membership (reverse sign for inverted conditions).
    """
    # Calibration constant b: at full_in, exp(b*(full_in-crossover)) / (1+...) ≈ 0.95
    # => b*(full_in-crossover) = ln(0.95/0.05) = ln(19)
    import math
    if abs(full_in - crossover) < 1e-12:
        raise ValueError('full_in cannot equal crossover')
    b = math.log(19) / (full_in - crossover)
    return 1.0 / (1.0 + np.exp(-b * (np.asarray(x, dtype=float) - crossover)))


# ── Load DSO tariff parameters ────────────────────────────────────────────────
dso = pd.read_csv(INPUTS / 'dso_tariffs_residential_2026.csv')
dso = dso.rename(columns={'DSO': 'dso_id'})

# MOD3_SIGNAL: §14a Modul 3 HT−NT spread [ct/kWh]; larger spread → stronger time signal
dso['MOD3_SIGNAL_raw'] = dso['HT_ct_kWh'] - dso['NT_ct_kWh']

# DSO_VOL_LEVEL: standard Arbeitspreis [ct/kWh]; higher rate → stronger volumetric dilution
dso['DSO_VOL_LEVEL_raw'] = dso['Arbeitspreis_ct_kWh']

# Calibration anchors derived from the empirical DSO distribution (N=7)
for col in ['MOD3_SIGNAL_raw', 'DSO_VOL_LEVEL_raw']:
    vals = dso[col]
    q20, q50, q80 = vals.quantile([0.20, 0.50, 0.80]).values
    print(f'{col}: p20={q20:.3f} | p50={q50:.3f} | p80={q80:.3f}')

# Anchors for MOD3_SIGNAL (high spread → higher membership)
mod3_p20, mod3_p50, mod3_p80 = dso['MOD3_SIGNAL_raw'].quantile([0.20,0.50,0.80]).values
dso['MOD3_SIGNAL'] = calibrate_fuzzy(
    dso['MOD3_SIGNAL_raw'], full_in=mod3_p80, crossover=mod3_p50, full_out=mod3_p20
).round(4)

# Anchors for DSO_VOL_LEVEL (high Arbeitspreis → higher membership)
vol_p20, vol_p50, vol_p80 = dso['DSO_VOL_LEVEL_raw'].quantile([0.20,0.50,0.80]).values
dso['DSO_VOL_LEVEL'] = calibrate_fuzzy(
    dso['DSO_VOL_LEVEL_raw'], full_in=vol_p80, crossover=vol_p50, full_out=vol_p20
).round(4)

print('\nDSO condition scores:')
print(dso[['dso_id','Arbeitspreis_ct_kWh','MOD3_SIGNAL_raw','MOD3_SIGNAL','DSO_VOL_LEVEL']].to_string(index=False))


MOD3_SIGNAL_raw: p20=8.074 | p50=8.560 | p80=11.810
DSO_VOL_LEVEL_raw: p20=5.638 | p50=6.400 | p80=7.548

DSO condition scores:
          dso_id  Arbeitspreis_ct_kWh  MOD3_SIGNAL_raw  MOD3_SIGNAL  DSO_VOL_LEVEL
        Westnetz                 9.53            14.70       0.9962         0.9997
      Bayernwerk                 4.72             8.56       0.5000         0.0133
           E.DIS                 5.47             8.25       0.4302         0.0843
        Netze BW                 7.57             8.03       0.3822         0.9526
Stromnetz Berlin                 7.46            11.33       0.9248         0.9381
         SH Netz                 6.40             7.68       0.3106         0.5000
   MITNETZ STROM                 6.31            11.93       0.9549         0.4425


## Step 5 — Assemble fsQCA truth-table input (N = 42 cases)

In [6]:
# ── Binary device conditions (from archetype number) ─────────────────────────
# Only include flexible archetypes (3–8); A1 and A2 have no NIR
fsqca_rows = wide[wide['household_archetype'].isin(range(3, 9))].copy()

fsqca_rows['BSS'] = fsqca_rows['household_archetype'].isin([3, 8]).astype(int)
fsqca_rows['HP']  = fsqca_rows['household_archetype'].isin([4, 6, 8]).astype(int)
fsqca_rows['EV']  = fsqca_rows['household_archetype'].isin([5, 7, 8]).astype(int)

# Join DSO-level conditions
fsqca_rows = fsqca_rows.merge(
    dso[['dso_id','Arbeitspreis_ct_kWh','MOD3_SIGNAL_raw','MOD3_SIGNAL','DSO_VOL_LEVEL']],
    on='dso_id', how='left'
)

# ── Calibrate LOW_NIR outcome ─────────────────────────────────────────────────
# Direction inverted: lower NIR → higher LOW_NIR membership
# Anchors from NIR distribution (Chapter 4, Section 4.8 / Appendix D)
nir_vals = fsqca_rows['NIR'].dropna()
nir_p25, nir_p50, nir_p75 = nir_vals.quantile([0.25, 0.50, 0.75]).values
print(f'NIR calibration anchors: full_in={nir_p25:.4f} | crossover={nir_p50:.4f} | full_out={nir_p75:.4f}')

# For inverted direction (low NIR → high LOW_NIR), pass negative values
fsqca_rows['LOW_NIR'] = calibrate_fuzzy(
    -fsqca_rows['NIR'],              # invert: low NIR becomes large
    full_in=-nir_p25,                # full member at low NIR (= high -NIR)
    crossover=-nir_p50,
    full_out=-nir_p75
).round(4)

# ── Case identifier ───────────────────────────────────────────────────────────
fsqca_rows['case_id'] = 'A' + fsqca_rows['household_archetype'].astype(str) + '_' + fsqca_rows['dso_id']

cols_out = ['case_id','household_archetype','dso_id',
            'BSS','HP','EV','MOD3_SIGNAL','DSO_VOL_LEVEL',
            'NIR','LOW_NIR',
            'NFI_eur','TCoE_noflex','TCoE_dtflex','TCoE_tcoeflex',
            'saving_dtflex_eur']
fsqca_out = fsqca_rows[cols_out].reset_index(drop=True)

print(f'\nfsQCA table: {len(fsqca_out)} rows (expected 42)')
display(fsqca_out.sort_values(['household_archetype','dso_id']))


NIR calibration anchors: full_in=0.4048 | crossover=0.4927 | full_out=0.5951

fsQCA table: 42 rows (expected 42)


,case_id,household_archetype,dso_id,BSS,HP,EV,MOD3_SIGNAL,DSO_VOL_LEVEL,NIR,LOW_NIR,NFI_eur,TCoE_noflex,TCoE_dtflex,TCoE_tcoeflex,saving_dtflex_eur
0,A3_Bayernwerk,3,Bayernwerk,1,0,0,0.5000,0.0133,0.5659,0.0790,177.44,11.58,-88.83,-165.86,100.41
1,A3_E.DIS,3,E.DIS,1,0,0,0.4302,0.0843,0.5303,0.2206,177.21,-6.84,-100.81,-184.05,93.97
2,A3_MITNETZ STROM,3,MITNETZ STROM,1,0,0,0.9549,0.4425,0.4903,0.5197,176.95,-2.64,-89.39,-179.59,86.75
3,A3_Netze BW,3,Netze BW,1,0,0,0.3822,0.9526,0.4299,0.8913,176.62,23.28,-52.65,-153.34,75.93
4,A3_SH Netz,3,SH Netz,1,0,0,0.3106,0.5000,0.4860,0.5555,176.93,24.34,-61.64,-152.59,85.98
5,A3_Stromnetz Berlin,3,Stromnetz Berlin,1,0,0,0.9248,0.9381,0.4352,0.8728,176.64,-38.11,-114.98,-214.75,76.87
6,A3_Westnetz,3,Westnetz,1,0,0,0.9962,0.9997,0.3353,0.9949,176.18,38.82,-20.26,-137.36,59.08
7,A4_Bayernwerk,4,Bayernwerk,0,1,0,0.5000,0.0133,0.7673,0.0001,193.10,2270.39,2122.23,2077.29,148.16
8,A4_E.DIS,4,E.DIS,0,1,0,0.4302,0.0843,0.4441,0.8359,331.70,2342.42,2195.12,2010.72,147.30
9,A4_MITNETZ STROM,4,MITNETZ STROM,0,1,0,0.9549,0.4425,0.4290,0.8942,341.11,2420.54,2274.21,2079.43,146.33


## Step 6 — QA checks

In [7]:
# 42 flexible cases
assert len(fsqca_out) == 42, f'Expected 42 cases, got {len(fsqca_out)}'

# Fuzzy scores in [0,1]
for col in ['MOD3_SIGNAL','DSO_VOL_LEVEL','LOW_NIR']:
    assert fsqca_out[col].between(0,1).all(), f'{col} out of [0,1]'

# LOW_NIR monotone: higher NIR → lower LOW_NIR
corr = fsqca_out['NIR'].corr(fsqca_out['LOW_NIR'])
assert corr < 0, f'LOW_NIR should be negatively correlated with NIR, got r={corr:.3f}'

# NIR within plausible range (no extreme values expected)
nir_check = fsqca_out['NIR'].dropna()
assert nir_check.between(-0.5, 2.0).all(), f'NIR out of expected range: {nir_check.describe()}'

# Binary conditions are 0 or 1
for col in ['BSS','HP','EV']:
    assert set(fsqca_out[col].unique()).issubset({0,1}), f'{col} has non-binary values'

print('QA passed: 42 cases, fuzzy [0,1], LOW_NIR inverted, binary conditions valid.')
print(f'NIR correlation with LOW_NIR: r = {corr:.4f}')


QA passed: 42 cases, fuzzy [0,1], LOW_NIR inverted, binary conditions valid.
NIR correlation with LOW_NIR: r = -0.8956


## Step 7 — Export

In [8]:
# Full NFI/NIR table (all 8 archetypes × 7 DSOs = 56 rows)
nfi_nir_full = wide[['household_archetype','dso_id',
                      'TCoE_noflex','TCoE_dtflex','TCoE_tcoeflex',
                      'NFI_eur',
                      'saving_dtflex_eur','NIR']].copy()
out1 = ANALYSIS / 'nfi_nir_germany_2026.csv'
nfi_nir_full.to_csv(out1, index=False)
print(f'Saved: {out1}  ({len(nfi_nir_full)} rows)')

# fsQCA conditions table (42 flexible cases)
out2 = ANALYSIS / 'fsqca_conditions_germany_2026.csv'
fsqca_out.to_csv(out2, index=False)
print(f'Saved: {out2}  ({len(fsqca_out)} rows)')

# Calibration anchor record (for reproducibility / Appendix D)
anchors = pd.DataFrame([
    {'condition':'LOW_NIR',       'full_in':nir_p25, 'crossover':nir_p50, 'full_out':nir_p75,  'direction':'inverted (low NIR = high membership)'},
    {'condition':'MOD3_SIGNAL',   'full_in':mod3_p80,'crossover':mod3_p50,'full_out':mod3_p20, 'direction':'direct (high spread = high membership)'},
    {'condition':'DSO_VOL_LEVEL', 'full_in':vol_p80, 'crossover':vol_p50, 'full_out':vol_p20,  'direction':'direct (high Arbeitspreis = high membership)'},
])
out3 = ANALYSIS / 'fsqca_calibration_anchors_germany_2026.csv'
anchors.to_csv(out3, index=False)
print(f'Saved: {out3}')
display(anchors)


Saved: /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub/notebooks/03_analysis/nfi_nir_germany_2026.csv  (56 rows)
Saved: /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub/notebooks/03_analysis/fsqca_conditions_germany_2026.csv  (42 rows)
Saved: /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub/notebooks/03_analysis/fsqca_calibration_anchors_germany_2026.csv


,condition,full_in,crossover,full_out,direction
0,LOW_NIR,0.404825,0.49265,0.595075,inverted (low NIR = high membership)
1,MOD3_SIGNAL,11.810000,8.56000,8.074000,direct (high spread = high membership)
2,DSO_VOL_LEVEL,7.548000,6.40000,5.638000,direct (high Arbeitspreis = high membership)
